In [1]:
# import all necessary libraries
import numpy as np
import pandas as pd
import h5py
import vaex
import pynbody
from pynbody.array import SimArray
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import PathPatch
from matplotlib.path import Path
from astropy import units as u
from astropy.io import ascii, fits
from astropy.table import Table, vstack
from astropy.coordinates import SkyCoord,CartesianRepresentation,match_coordinates_sky
import functions_altered as fn 
from matched_filter_ani_altered import matched_filter_ani as mf
import os
from pathlib import Path as path

np.random.seed(0)

ModuleNotFoundError: No module named 'functions_altered'

In [3]:
s = pynbody.load('romulus_zooms/r741.romulus25.3072g1HsbBH/r741.romulus25.3072g1HsbBH.004096/r741.romulus25.3072g1HsbBH.004096')

h = s.halos(halo_numbers='v1')  # Load the halos using the original AHF numbering system
h.load_all()
unique_halo_ids = list(h.keys())


In [4]:
num = 741 
main_halo = h[1] ### IF WORKING WITH MARVEL, CHANGE TO h[num].


In [5]:
pynbody.analysis.halo.center
cen = main_halo.mean_by_mass('pos')

sp = s[pynbody.filt.Sphere(SimArray([200], "kpc"), cen)].load_copy()

s.physical_units()


with h5py.File('romulus_zooms/r741.romulus25.3072g1HsbBH/r741_allhalostardata_consolidated2.h5','r') as f:
    hostids = f['host_IDs'].asstr()[:] 
    partids_h5 = f['particle_IDs'][:]

partids_snap = sp.s['iord']

In [6]:
box = 'r' 
D = 2000 
mlim_str = '26p5' 
name = f'{box}_4096_{num}_data_{D}_{mlim_str}'

#dwarfcatpath = f"/home/otteleno/MAP/matched_filter_starters/{box}_{num}_data/survey.{name}.0.h5" ###MARVEL CONVENTION
dwarfcatpath = f"/home/otteleno/MAP/matched_filter_starters/{box}_{num}_data/{box}{num}/survey.MM_r741_data_2000_26p5.0.h5" ###MM CONVENTION
vdwarfcat = vaex.open(dwarfcatpath)
dwarfcat = pd.DataFrame(vdwarfcat,columns=vdwarfcat.column_names)

size_kpc = 40 
pdist = (size_kpc/D) * (180/np.pi) 
year = 10
mlim = '25'
c1='px'
c2='py'
edgelength = 10 
plotdir = f'{box}_4096_{num}' 

In [7]:
dwarfcat

,age,dec,dmod,feh,glat,glon,grav,lsst_g,lsst_g_Err,lsst_g_Intrinsic,...,py,pz,ra,rad,smass,teff,vr,vx,vy,vz
0,10.142102,-27.406462,26.498549,-40.020882,-89.614544,-13.337249,0.732200,24.518137,-0.0,-1.980411,...,-3.094372,-1993.884098,12.559338,1993.929219,0.787524,4511.346680,99.506002,20.553106,-229.181537,-99.018037
1,10.141779,-27.105970,26.504638,-4.013334,-89.961931,68.750085,1.805265,26.257990,0.0,-0.246647,...,1.238220,-1999.528219,12.824800,1999.528660,0.787121,4962.934570,72.860977,19.446306,-221.307146,-72.993356
2,10.141671,-27.407793,26.498072,-3.177445,-89.607567,-12.563417,2.087764,26.800455,-0.0,0.302382,...,-2.969982,-1993.445072,12.549614,1993.491832,0.786821,5070.593750,97.208341,28.923647,-232.565814,-96.670762
3,10.141644,-27.404403,26.498319,-3.205892,-89.614048,-12.822129,2.116079,26.856220,0.0,0.357902,...,-2.980418,-1993.672667,12.556145,1993.717900,0.786786,5081.625977,88.262891,20.024706,-229.214758,-87.790704
4,10.141644,-27.159282,26.503470,-2.602483,-89.964558,-85.957445,2.047623,26.726347,0.0,0.222878,...,-1.233113,-1998.452781,12.878722,1998.453163,0.786868,5055.708984,75.442880,2.172189,-185.524267,-75.328325
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282209,4.770066,-27.095193,26.502154,-0.845207,-89.919793,-171.390956,4.332410,23.661863,-0.0,-2.840289,...,-0.418519,-1997.240117,12.941575,1997.242074,12.770525,31380.939453,93.794032,-14.264906,-165.010287,-93.739802
282210,4.770066,-27.095179,26.502160,-0.845207,-89.920208,-171.536808,4.354091,23.937067,-0.0,-2.565094,...,-0.409357,-1997.245869,12.941057,1997.247805,11.492317,30024.656250,93.709803,-13.892272,-164.554310,-93.657030
282211,4.770066,-27.095429,26.502188,-0.845207,-89.920720,-171.506641,4.373773,24.206783,-0.0,-2.295404,...,-0.408174,-1997.271582,12.940553,1997.273494,10.358454,28728.912109,93.692574,-14.282147,-164.381196,-93.639524
282212,4.770066,-27.095042,26.502175,-0.845207,-89.919988,-171.572726,4.165921,22.205545,0.0,-4.296630,...,-0.408757,-1997.259954,12.941258,1997.261901,23.179908,38428.832031,93.362568,-14.113036,-164.878743,-93.309420


In [8]:
ananke_data = np.column_stack([dwarfcat['px'],dwarfcat['py'], dwarfcat['pz']])
ananke_masses = dwarfcat['smass']
ananke_total_mass = np.sum(ananke_masses)

cm_current_ananke = [0,0,0]
for i in range(len(ananke_data)):
    cm_current_ananke = cm_current_ananke + ananke_data[i] * ananke_masses[i]
cm_ananke = [x/ananke_total_mass for x in cm_current_ananke]


ananke_data_centered = ananke_data - cm_ananke
print(ananke_data_centered)

[[13.2010265  -3.20805588  6.1849746 ]
 [ 0.63033572  1.12453577  0.54085319]
 [13.47572094 -3.08366623  6.624     ]
 ...
 [-2.58450537 -0.52185848  2.79749009]
 [-2.61018892 -0.5224412   2.80911848]
 [-2.60853654 -0.51595123  2.80817815]]


In [11]:
def fibonacci_angler(n):

    golden_ratio = (1 + np.sqrt(5))/2

    i_array = np.linspace(0,n-1,n)
    z_array = 1 - i_array/((n-1)) ###ONLY GOES HALF WAY DOWN
    radius_array = np.sqrt(1-z_array**2)

    declination_array = np.pi/2-np.arcsin(z_array)
    azimuthal_array = 2*np.pi * i_array/golden_ratio
    

    return declination_array, azimuthal_array

In [12]:
declination_array, azimuthal_array = fibonacci_angler(24)

In [13]:
def new_axes(host_ids):

    #Calculates statistics for the entire galaxy
    
    m = sp.s['mass']
    x = sp.s['pos'][:, 0]
    y = sp.s['pos'][:, 1]
    z = sp.s['pos'][:, 2]
    vx = sp.s['vel'][:, 0]
    vy = sp.s['vel'][:, 1]
    vz = sp.s['vel'][:, 2]

    particle_array_all = np.column_stack((m,x,y,z,vx,vy,vz))
    pos_array_all = particle_array_all[:, 1:4]
    vel_array_all = particle_array_all[:, 4:7]
    
    total_mass_all=np.sum(m)
    
    cm_current_all = [0,0,0]
    for i in range(len(particle_array_all)):
        cm_current_all = cm_current_all + particle_array_all[i][0] * pos_array_all[i]
    cm_all = cm_current_all/total_mass_all
    
    avg_vel_current_all = [0,0,0]
    for i in range(len(particle_array_all)):
        avg_vel_current_all = avg_vel_current_all + particle_array_all[i][0] * vel_array_all[i]
    avg_vel_all = avg_vel_current_all/total_mass_all
    
    pos_array_centered_all = pos_array_all - cm_all
    vel_array_centered_all = vel_array_all - avg_vel_all


    #Calculates statistics only for the halo of interest, but adjusts them using the entire galaxy
    
    _, idloc_snap, idloc_h5 = np.intersect1d(partids_snap, partids_h5, return_indices = True) 
                                                                                             
    
    progenitor = hostids[idloc_h5] 
    mask = np.isin(progenitor, host_ids) 
    
    m_halo = m[idloc_snap][mask] 
    x_halo = x[idloc_snap][mask]
    y_halo = y[idloc_snap][mask]
    z_halo = z[idloc_snap][mask]
    vx_halo = vx[idloc_snap][mask]
    vy_halo = vy[idloc_snap][mask]
    vz_halo = vz[idloc_snap][mask]

    particle_array_halo = np.column_stack((m_halo,x_halo,y_halo,z_halo,vx_halo,vy_halo,vz_halo)) 
    pos_array_halo = particle_array_halo[:, 1:4]
    vel_array_halo = particle_array_halo[:, 4:7]
    pos_array_centered_halo = pos_array_halo - cm_all 
    vel_array_centered_halo = vel_array_halo - avg_vel_all 
    
    total_mass_halo=np.sum(m_halo)

    cm_current_halo = [0,0,0]
    for i in range(len(particle_array_halo)):
        cm_current_halo = cm_current_halo + particle_array_halo[i][0] * pos_array_centered_halo[i]
    cm_halo = [x/total_mass_halo for x in cm_current_halo]
    
    ang_mom_halo = [0,0,0]
    for i in range(len(particle_array_halo)):
        ang_mom_halo = ang_mom_halo + particle_array_halo[i][0] * np.cross(pos_array_centered_halo[i], vel_array_centered_halo[i])

    #Creates new basis and transforms data

    z_prime = ang_mom_halo/np.linalg.norm(ang_mom_halo)
    y_prime_unnormed = np.cross(ang_mom_halo, cm_halo)
    y_prime =  y_prime_unnormed/np.linalg.norm(y_prime_unnormed)
    x_prime = np.cross(z_prime, y_prime)
    transform_matrix = np.row_stack((x_prime, y_prime, z_prime)) 
    
    return transform_matrix
    
    

In [14]:
transform_matrix = new_axes('1248_4')

In [15]:
print(transform_matrix)

[[-0.22727139 -0.42754771  0.87495753]
 [ 0.89506909 -0.44568474  0.01471169]
 [-0.38366527 -0.78649098 -0.48397613]]


In [16]:
def angle_view(coordinate_transform, declination, azimuthal,save = False, plot=True):

    '''Calculates 2d image coordinates based on angle of view. Plots graph. Many optional parameters for moviemaking ease

    host_ids = hostid(s) of halo to calibrate axes around, as well as color differently
    declination = camera angle measured down from z axis
    azimuthal = camera angle measured counterclockwise from x axis in xy plane
    
    Below are largely movie features 
    
    plot = Whether to display plot on Jupyter or not.
    save = Whether to save plot to file system
    axis_of_rotation = x,y, or z. Axis camera is rotation around.
    angle = Angle from current camera position to default camera position (depends on what type of movie you're making
    limits = 1x4 array of [min image(x), max image(x), min image(y), max image(y)]
    n = index of what image. Stored like 000 for first image, and 010 for 11th image for example.
    
    '''
    #Transforms data into new axes
    new_pos_array_all = np.matmul(transform_matrix, ananke_data_centered.T)

    #Spherical Coordinates
    theta = declination
    phi = azimuthal 
    declination_readable = round(declination*180/np.pi,1)
    azimuthal_readable = round(np.mod(azimuthal*180/np.pi, 360),1)

    #Uses generalized matrix to find 2d projected image from any given camera angle 
    project_matrix = np.array([[-np.sin(phi),                np.cos(phi),             0            ],
                             [-np.cos(theta)*np.cos(phi), -np.cos(theta)*np.sin(phi), np.sin(theta)]])

    project_pos_array_all = np.matmul(project_matrix, new_pos_array_all) #applies 2x3 transformation to 3xn data. Result is 2xn data 

    if plot == True:
        fig,ax  = plt.subplots()
        ax.scatter (project_pos_array_all[0], project_pos_array_all[1], s=1)
        ax.set_title(f"Declination = {declination_readable} degrees and Azimuthal = {azimuthal_readable} degrees")
        ax.set_title
        plt.show()
       

    return declination_readable, azimuthal_readable, project_pos_array_all



In [18]:
for i in range(len(declination_array)):
    d, a, altered_positions=  angle_view(transform_matrix, declination_array[i], azimuthal_array[i], plot=False)
    dwarfcat['px'] = altered_positions.T[:,0]
    dwarfcat['py'] = altered_positions.T[:,1]

    file_path = f"/home/otteleno/MAP/automated_data/rotated_catalogs/{box}_{num}/{box}_{num}_d={d}_a={a}.h5"
    if os.path.exists(file_path):
        os.remove(file_path)

    vframe_new = vaex.from_pandas(dwarfcat)
    vframe_new.export_hdf5(file_path, progress=True)
    






export(hdf5) [########################################] 100.00% elapsed time  :     2.34s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     2.79s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     2.58s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     2.77s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     2.77s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     2.76s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     2.64s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     4.27s =  0.1m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     4.57s =  0.1m =  0.0h
export(hdf5) [################################

In [18]:
print(len(ananke_data))
print(len(sp))

276389
17074495
